# Score por merchant y join a las transacciones

Paso 2 de la consigna: cada transacción lleva el score de riesgo de su merchant.

- **Score:** `data/processed/score_octubre.parquet`, de `05_logistica.ipynb`. Una fila por merchant, con la logística B ajustada con las etiquetas de abril a septiembre. La fila es la del 1-oct, que resume la actividad hasta el 30-sep.
- **Join:** por `merchant_id`. Las transacciones no tienen fecha, así que un merchant tiene un único score para todas sus transacciones de octubre a diciembre.
- **`risk_pct`:** el percentil del score entre todos los merchants (0 = menor riesgo, 1 = mayor). El score de la logística no está calibrado (se entrena con pesos balanceados), así que los umbrales de las reglas se definen por percentil y no por probabilidad.
- **`is_fraud` y `approved` no entraron al modelo.** Quedan intactos para medir el impacto de negocio.

Salida: `data/processed/transacciones_con_score.parquet`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent
TXN_PATH = ROOT / "data" / "txn_sample.parquet"
SCORE_PATH = ROOT / "data" / "processed" / "score_octubre.parquet"
OUT_PATH = ROOT / "data" / "processed" / "transacciones_con_score.parquet"

txn = pd.read_parquet(TXN_PATH)
score = pd.read_parquet(SCORE_PATH)
assert txn["transaction_id"].is_unique
assert score["merchant_id"].is_unique
print(f"{len(txn):,} transacciones, {txn['merchant_id'].nunique():,} merchants")
print(f"{len(score):,} merchants con score (fila del {score['observation_date'].mode().iloc[0].date()} para {int((score['observation_date'] == score['observation_date'].mode().iloc[0]).sum()):,})")

## Percentil del score

Se calcula sobre todos los merchants con score, no sobre las transacciones: así un merchant grande no pesa más en el ranking.

In [ ]:
score["risk_pct"] = score["score"].rank(method="average", pct=True)
print(f"scores distintos: {score['score'].nunique():,} de {len(score):,}")
top_tie = score["score"].value_counts().head(3)
print("valores más repetidos del score:")
print(top_tie.round(4).to_string())
print(score["score"].describe().round(4).to_string())

## Join y cobertura

In [ ]:
scored = txn.merge(score[["merchant_id", "score", "risk_pct"]], on="merchant_id", how="left", validate="many_to_one")
assert len(scored) == len(txn)

no_score = scored["score"].isna()
print(f"transacciones sin score: {int(no_score.sum()):,} ({no_score.mean():.2%})")
print(f"merchants sin score:     {scored.loc[no_score, 'merchant_id'].nunique():,}")
print(f"merchants con score que no aparecen en las transacciones: {len(set(score['merchant_id']) - set(txn['merchant_id'])):,}")
assert not no_score.any(), "hay transacciones sin score"
print(f"amount: {scored['amount'].isna().sum():,} nulos | approved: {scored['approved'].value_counts().to_dict()} | is_fraud: {scored['is_fraud'].sum():,}")

## Cómo se reparten las transacciones por nivel de riesgo

Es una descripción, no una evaluación: `is_fraud` y `approved` no se usaron para entrenar, así que sirven de chequeo de sentido. La evaluación de las reglas es el paso siguiente.

In [ ]:
bins = [0, 0.50, 0.90, 0.95, 0.99, 1.0001]
labels = ["0-50%", "50-90%", "90-95%", "95-99%", "99-100%"]
scored["tramo"] = pd.cut(scored["risk_pct"], bins=bins, labels=labels, right=False)
merchants = scored.drop_duplicates("merchant_id")
table = scored.groupby("tramo", observed=True).agg(
    transacciones=("transaction_id", "size"),
    tpv=("amount", "sum"),
    aprobacion=("approved", "mean"),
    fraude=("is_fraud", "mean"),
    fraudes=("is_fraud", "sum"),
)
table.insert(0, "merchants", merchants.groupby("tramo", observed=True).size())
table["% transacciones"] = table["transacciones"] / table["transacciones"].sum()
table["% tpv"] = table["tpv"] / table["tpv"].sum()
print(table.round({"aprobacion": 3, "fraude": 5, "% transacciones": 4, "% tpv": 4}).to_string())

## Se deja documentado

La tabla y el gráfico se guardan en `docs/score/` y se citan en `docs/decisions.md` (sección 4).

In [ ]:
DOCS = ROOT / "docs" / "score"
DOCS.mkdir(parents=True, exist_ok=True)
out = table.copy()
out["% fraudes"] = out["fraudes"] / out["fraudes"].sum()
out.round(6).to_csv(DOCS / "riesgo_por_tramo.csv", index_label="tramo")

import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))
x = np.arange(len(out))
w = 0.27
ax[0].bar(x - w, out["% transacciones"] * 100, w, label="% transacciones", color="#B4B2A9")
ax[0].bar(x, out["% tpv"] * 100, w, label="% TPV", color="#378ADD")
ax[0].bar(x + w, out["% fraudes"] * 100, w, label="% fraudes", color="#D85A30")
ax[0].set_xticks(x, out.index)
ax[0].set_ylabel("% del total")
ax[0].set_title("Qué parte de transacciones, TPV y fraudes cae en cada tramo")
ax[0].legend()
ax[1].bar(x, out["fraude"] * 100, color="#D85A30")
ax[1].axhline(scored["is_fraud"].mean() * 100, color="#444441", ls="--", label="promedio general")
for i, v in enumerate(out["fraude"] * 100):
    ax[1].text(i, v + 0.002, f"{v:.3f}%", ha="center", fontsize=9)
ax[1].set_xticks(x, out.index)
ax[1].set_ylabel("tasa de fraude (%)")
ax[1].set_title("Tasa de fraude por tramo de riesgo del merchant")
ax[1].legend()
for a in ax:
    a.set_xlabel("percentil de riesgo del merchant")
fig.tight_layout()
fig.savefig(DOCS / "riesgo_por_tramo.png", dpi=110)
plt.close(fig)
print("guardado", DOCS)

In [ ]:
OUT_PATH.parent.mkdir(exist_ok=True)
final = scored.drop(columns="tramo")
final.to_parquet(OUT_PATH, index=False)
print(f"guardado {OUT_PATH.name}: {len(final):,} filas, {OUT_PATH.stat().st_size / 1e6:.0f} MB")
print(list(final.columns))